<a href="https://colab.research.google.com/github/Gopal2210G/KodeinKGP-Submissions/blob/react/chapter_appendix-tools-for-deep-learning/jupyter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# 🧠 Llama 3 8B Instruct Fine-tuning (QLoRA) - Next Scene Prediction
# ============================================================

!pip install -q git+https://github.com/huggingface/transformers.git@main
!pip install -q git+https://github.com/huggingface/peft.git
!pip install -q bitsandbytes accelerate datasets trl sentencepiece safetensors

# ============================================================
# 🔧 Configuration
# ============================================================

import json, torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import os

# CHANGE THESE
MODEL_NAME = "meta-llama/Meta-Llama-3-8B-Instruct"   # or smaller if limited GPU
HF_TOKEN = "hf_xxx_your_token_here"                   # your Hugging Face token
DATA_PATH = "/content/next_scene_dataset.jsonl"       # upload file to /content first
OUTPUT_DIR = "/content/llama3_nextscene_qlora"

# ============================================================
# 🧱 Step 1: Load + Reformat Dataset
# ============================================================

print("Loading and formatting dataset...")

src = DATA_PATH
out_path = "/content/next_scene_dataset_instr.jsonl"

records = []
with open(src, "r", encoding="utf-8") as f:
    first = f.read(1); f.seek(0)
    data = json.load(f) if first == "[" else [json.loads(l) for l in f if l.strip()]

for rec in data:
    desc_key = next((k for k in rec if "description" in k.lower()), None)
    next_key = next((k for k in rec if "next" in k.lower()), None)
    if not desc_key or not next_key: continue
    records.append({
        "instruction": "Predict the next probable scene description given the current scene.",
        "input": rec[desc_key].strip(),
        "output": rec[next_key].strip()
    })

with open(out_path, "w", encoding="utf-8") as f:
    for r in records: f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"✅ Reformatted {len(records)} samples")

# ============================================================
# 🧾 Step 2: Load Dataset and Tokenizer
# ============================================================

dataset = load_dataset("json", data_files=out_path)["train"]
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_auth_token=HF_TOKEN)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

def format_example(ex):
    prompt = (
        f"### Instruction:\n{ex['instruction']}\n\n"
        f"### Input:\n{ex['input']}\n\n"
        f"### Response:\n"
    )
    return {"text": prompt + ex["output"] + tokenizer.eos_token}

dataset = dataset.map(format_example)

def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True, max_length=1024)
tokenized_ds = dataset.map(tokenize_fn, batched=True, remove_columns=dataset.column_names)

# ============================================================
# ⚙️ Step 3: Load Model in 4-bit and Add QLoRA
# ============================================================

print("Loading model in 4-bit... this may take a few minutes ⏳")

bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=bnb_cfg,
    use_auth_token=HF_TOKEN
)

model = prepare_model_for_kbit_training(model)

lora_cfg = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj","k_proj","v_proj","o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_cfg)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

# ============================================================
# 🏋️‍♂️ Step 4: Training
# ============================================================

def data_collator(features):
    return tokenizer.pad(features, padding=True, return_tensors="pt")

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=10,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
    optim="paged_adamw_8bit",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds,
    data_collator=data_collator,
    tokenizer=tokenizer
)

print("🚀 Starting training ...")
trainer.train()
trainer.save_model(OUTPUT_DIR)
print("✅ Training complete. Model saved at", OUTPUT_DIR)

# ============================================================
# 💬 Step 5: Inference (Test Generation)
# ============================================================

from transformers import pipeline
from peft import PeftModel

# reload base model + trained adapter
print("Loading model for inference...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=bnb_cfg,
    use_auth_token=HF_TOKEN
)
model = PeftModel.from_pretrained(model, OUTPUT_DIR)

generator = pipeline("text-generation", model=model, tokenizer=tokenizer)

def predict_next_scene(description):
    prompt = (
        "### Instruction:\nPredict the next probable scene description given the current scene.\n\n"
        f"### Input:\n{description}\n\n### Response:\n"
    )
    res = generator(prompt, max_new_tokens=120, temperature=0.4, do_sample=True)
    return res[0]["generated_text"].split("### Response:")[-1].strip()

# 🔍 Example Prediction
desc = "A view from behind of a roofer wearing an orange safety vest and fall protection harness."
print("\n🪄 Next scene prediction:\n", predict_next_scene(desc))

# ============================================================
# 💾 Step 6: Save lightweight adapter
# ============================================================

model.save_pretrained("/content/llama3_nextscene_adapter")
tokenizer.save_pretrained("/content/llama3_nextscene_adapter")
print("Adapter saved at /content/llama3_nextscene_adapter ✅")
